# Scaling Patterns: Multiple Products and Cross-Validation

**Goal:** Handle all bakery products simultaneously and use cross-validation to make a defensible model selection decision.

**What you will do:**
1. Train a global model across all unique_ids at once
2. Batch-process forecasts for large catalogs
3. Run `cross_validation` to compare NHITS vs a baseline
4. Profile training time vs accuracy trade-offs
5. Generate a production-ready model selection report

**Time:** ~13 minutes

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, MLP
from neuralforecast.losses.pytorch import MQLoss, MAE
from utilsforecast.losses import smape, mase
from utilsforecast.evaluation import evaluate

print('Libraries loaded.')
import neuralforecast
print(f'neuralforecast: {neuralforecast.__version__}')

## Step 1: Load All Products

The French Bakery dataset has multiple products. We train one global model on all of them.

**Why global models work:** All products in the same bakery share weekly seasonality — fewer sales Monday, more Friday/Saturday. A global model learns this pattern from thousands of data points across all products, not from each product's limited individual history.

In [ ]:
URL = (
    'https://raw.githubusercontent.com/nixtla/transfer-learning-time-series'
    '/main/datasets/french_bakery/bakery.csv'
)
raw = pd.read_csv(URL, parse_dates=['date'])

# Convert to nixtla format — keep all products
df_all = (
    raw.rename(columns={'date': 'ds', 'Quantity': 'y', 'article': 'unique_id'})
    [['unique_id', 'ds', 'y']]
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

products = df_all['unique_id'].unique().tolist()
print(f'Products ({len(products)}): {products}')

# Summary statistics per product
summary = (
    df_all.groupby('unique_id')['y']
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .round(1)
)
print('\nPer-product statistics:')
print(summary.to_string())

In [ ]:
# Visualize all products
n_products = len(products)
fig, axes = plt.subplots(n_products, 1, figsize=(12, 2.5 * n_products), sharex=True)

if n_products == 1:
    axes = [axes]

for ax, product in zip(axes, products):
    series = df_all[df_all['unique_id'] == product]
    ax.plot(series['ds'], series['y'], linewidth=0.8, alpha=0.8)
    ax.set_title(product, fontsize=10)
    ax.set_ylabel('Units')
    ax.grid(alpha=0.3)

plt.suptitle('All Bakery Products — Daily Demand', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Step 2: Train a Global Model

One `nf.fit()` call trains on all products simultaneously.
NeuralForecast handles series of different lengths automatically.

In [ ]:
H = 7
QUANTILES = [0.1, 0.5, 0.8, 0.9]

# Time the training
t0 = time.time()

model = NHITS(
    h=H,
    input_size=3 * H,
    loss=MQLoss(quantiles=QUANTILES),
    max_steps=300,
    random_seed=42,
)

nf = NeuralForecast(models=[model], freq='D')
nf.fit(df_all)  # trains on ALL products at once

train_time = time.time() - t0
print(f'\nGlobal model trained in {train_time:.1f} seconds.')
print(f'One model covers {len(products)} products.')

## Step 3: Generate Forecasts for All Products

`.predict()` returns forecasts for all unique_ids in one call.
For very large catalogs (10,000+ series), we batch by unique_id to manage memory.

In [ ]:
# Standard prediction: all products in one call
t0 = time.time()
forecast_all = nf.predict()
predict_time = time.time() - t0

print(f'Forecasted {forecast_all["unique_id"].nunique()} products in {predict_time:.2f}s')
print(forecast_all.to_string())

In [ ]:
# Batch prediction function (for large catalogs)
def forecast_in_batches(
    nf: NeuralForecast,
    all_series_df: pd.DataFrame,
    batch_size: int = 2,  # small for demo; use 500+ in production
) -> pd.DataFrame:
    """
    Generate forecasts for a large catalog in batches.

    Training is still global (one fit call).
    Only prediction is batched to manage memory.
    """
    unique_ids = all_series_df['unique_id'].unique().tolist()
    all_forecasts = []
    n_batches = (len(unique_ids) + batch_size - 1) // batch_size

    for i, start in enumerate(range(0, len(unique_ids), batch_size)):
        batch_ids = unique_ids[start: start + batch_size]
        batch_df = all_series_df[all_series_df['unique_id'].isin(batch_ids)]
        batch_fc = nf.predict(df=batch_df)
        all_forecasts.append(batch_fc)
        print(f'  Batch {i+1}/{n_batches}: {batch_ids}')

    return pd.concat(all_forecasts, ignore_index=True)


print('Running batch prediction demo (batch_size=2):')
forecast_batched = forecast_in_batches(nf, df_all, batch_size=2)

# Verify: batched and non-batched forecasts match
forecast_all_sorted = forecast_all.sort_values(['unique_id', 'ds']).reset_index(drop=True)
forecast_batched_sorted = forecast_batched.sort_values(['unique_id', 'ds']).reset_index(drop=True)

q_cols = [c for c in forecast_all.columns if 'q-' in c]
max_diff = (forecast_all_sorted[q_cols] - forecast_batched_sorted[q_cols]).abs().max().max()
print(f'\nMax difference between batched and non-batched: {max_diff:.6f} (should be ~0)')

## Step 4: 80% Service Level Orders for All Products

In [ ]:
q80_col = [c for c in forecast_all.columns if 'q-0.8' in c][0]
q50_col = [c for c in forecast_all.columns if 'q-0.5' in c][0]

# Weekly order per product at 80% SL
order_summary = (
    forecast_all.groupby('unique_id')[[q50_col, q80_col]]
    .sum()
    .apply(np.ceil)
    .astype(int)
    .rename(columns={q50_col: 'median_order', q80_col: 'sl80_order'})
)
order_summary['safety_stock'] = order_summary['sl80_order'] - order_summary['median_order']
order_summary['safety_pct'] = (
    order_summary['safety_stock'] / order_summary['median_order'] * 100
).round(1)

print('Weekly order recommendations (80% service level):')
print(order_summary.to_string())

# Visualization
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(order_summary))
width = 0.35

ax.bar(x, order_summary['median_order'], width, label='Median order', color='steelblue')
ax.bar(
    x, order_summary['safety_stock'], width,
    bottom=order_summary['median_order'],
    label='Safety stock (80% SL buffer)', color='orange', alpha=0.8
)

ax.set_xticks(x)
ax.set_xticklabels(order_summary.index, rotation=30, ha='right')
ax.set_title('Weekly Orders: Median vs 80% Service Level')
ax.set_ylabel('Units (7-day total)')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Step 5: Cross-Validation for Model Selection

`nf.cross_validation()` performs a walk-forward validation:
- Train on the first N steps
- Forecast H steps ahead
- Advance the cutoff by `step_size` steps
- Repeat for `n_windows` windows

This gives us an honest estimate of how the model performs on unseen data — which is exactly what production performance looks like.

In [ ]:
# Compare NHITS vs MLP baseline via cross-validation
# n_windows=3 gives 3 evaluation windows — enough for a quick comparison

models_to_compare = [
    NHITS(
        h=H,
        input_size=3 * H,
        loss=MQLoss(quantiles=QUANTILES),
        max_steps=200,  # reduced for speed in demo
        random_seed=42,
        alias='NHITS',
    ),
    MLP(
        h=H,
        input_size=3 * H,
        loss=MQLoss(quantiles=QUANTILES),
        max_steps=200,
        random_seed=42,
        alias='MLP_baseline',
    ),
]

nf_cv = NeuralForecast(models=models_to_compare, freq='D')

t0 = time.time()
cv_results = nf_cv.cross_validation(
    df=df_all,
    n_windows=3,       # number of evaluation windows
    step_size=H,       # advance cutoff by 7 days each time
)
cv_time = time.time() - t0

print(f'Cross-validation completed in {cv_time:.1f} seconds.')
print(f'CV results shape: {cv_results.shape}')
print('Columns:', cv_results.columns.tolist())
print(cv_results.head(7))

In [ ]:
# Compute SMAPE per model using utilsforecast
# We evaluate on the median (0.5 quantile) predictions

nhits_q50 = [c for c in cv_results.columns if 'NHITS' in c and 'q-0.5' in c]
mlp_q50 = [c for c in cv_results.columns if 'MLP_baseline' in c and 'q-0.5' in c]

print('NHITS median columns:', nhits_q50)
print('MLP median columns:', mlp_q50)

if nhits_q50 and mlp_q50:
    # Compute sMAPE for each model
    def compute_smape(cv_df, pred_col, actual_col='y'):
        """Symmetric mean absolute percentage error."""
        y = cv_df[actual_col].values
        yhat = cv_df[pred_col].values
        return 200 * np.mean(np.abs(y - yhat) / (np.abs(y) + np.abs(yhat) + 1e-8))

    smape_nhits = compute_smape(cv_results, nhits_q50[0])
    smape_mlp = compute_smape(cv_results, mlp_q50[0])

    print(f'\nsMAPE — NHITS: {smape_nhits:.2f}%')
    print(f'sMAPE — MLP:   {smape_mlp:.2f}%')

    winner = 'NHITS' if smape_nhits < smape_mlp else 'MLP'
    improvement = abs(smape_nhits - smape_mlp)
    print(f'\nRecommended model: {winner}')
    print(f'Improvement over alternative: {improvement:.2f} pp sMAPE')

In [ ]:
# Plot actual vs predicted for BAGUETTE across CV windows
baguette_cv = cv_results[cv_results['unique_id'] == 'BAGUETTE'].copy()

if nhits_q50 and len(baguette_cv) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))

    # Group by cutoff (each CV window)
    cutoffs = baguette_cv['cutoff'].unique()
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(cutoffs)))

    for cutoff, color in zip(cutoffs, colors):
        window = baguette_cv[baguette_cv['cutoff'] == cutoff]
        ax.plot(window['ds'], window['y'], color='steelblue', alpha=0.5,
                linewidth=1, linestyle='--')
        ax.plot(window['ds'], window[nhits_q50[0]], color=color, linewidth=1.5,
                label=f'NHITS (cutoff {cutoff.date()})')

    ax.set_title('BAGUETTE: Cross-Validation Windows — Actual vs NHITS Median Forecast')
    ax.set_ylabel('Units')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

else:
    print('CV results available — modify the column names above if needed.')

## Step 6: Profile Training Time vs Accuracy

Training time scales with `max_steps`. This profile shows the accuracy-compute trade-off
so you can choose the minimum compute budget that meets accuracy requirements.

In [ ]:
# Profile max_steps vs sMAPE on baguette
# We use a single product to keep this fast
baguette_only = df_all[df_all['unique_id'] == 'BAGUETTE'].copy()

step_budgets = [50, 100, 200, 400]  # MODIFY: try different values
results = []

for steps in step_budgets:
    t0 = time.time()

    m = NHITS(
        h=H,
        input_size=3 * H,
        loss=MQLoss(quantiles=QUANTILES),
        max_steps=steps,
        random_seed=42,
    )
    nf_profile = NeuralForecast(models=[m], freq='D')
    cv = nf_profile.cross_validation(df=baguette_only, n_windows=2, step_size=H)

    elapsed = time.time() - t0

    q50_col = [c for c in cv.columns if 'q-0.5' in c][0]
    smape_val = compute_smape(cv, q50_col)

    results.append({'max_steps': steps, 'time_s': elapsed, 'smape': smape_val})
    print(f'max_steps={steps:>4}: {elapsed:.1f}s, sMAPE={smape_val:.2f}%')

profile_df = pd.DataFrame(results)
print('\nProfile summary:')
print(profile_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: sMAPE vs max_steps
axes[0].plot(profile_df['max_steps'], profile_df['smape'],
             marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Accuracy vs Training Budget')
axes[0].set_xlabel('max_steps')
axes[0].set_ylabel('sMAPE (%)')
axes[0].grid(alpha=0.3)

# Right: Training time vs max_steps
axes[1].plot(profile_df['max_steps'], profile_df['time_s'],
             marker='o', linewidth=2, color='orange')
axes[1].set_title('Training Time vs Training Budget')
axes[1].set_xlabel('max_steps')
axes[1].set_ylabel('Time (seconds)')
axes[1].grid(alpha=0.3)

plt.suptitle('NHITS: Accuracy–Compute Trade-off (BAGUETTE)', fontsize=12)
plt.tight_layout()
plt.show()

# Find the elbow: where accuracy gains flatten
smape_vals = profile_df['smape'].values
gains = np.diff(smape_vals)  # negative = improvement
print('\nAccuracy gain per 100-step increment:')
for i, gain in enumerate(gains):
    print(f'  {profile_df["max_steps"].iloc[i]}→{profile_df["max_steps"].iloc[i+1]}: {gain:+.3f}% sMAPE')

## Step 7: Production Model Selection Report

Aggregate the CV results into a decision-ready report.

In [ ]:
# Build a production model selection report
def model_selection_report(
    cv_results: pd.DataFrame,
    model_prefixes: list[str],
    train_time_seconds: float,
) -> pd.DataFrame:
    """
    Summarize cross-validation results across models.

    Returns a DataFrame with sMAPE per model per product.
    """
    records = []

    for model_prefix in model_prefixes:
        q50_cols = [c for c in cv_results.columns if model_prefix in c and 'q-0.5' in c]
        if not q50_cols:
            continue
        q50_col = q50_cols[0]

        for uid in cv_results['unique_id'].unique():
            uid_cv = cv_results[cv_results['unique_id'] == uid]
            s = compute_smape(uid_cv, q50_col)
            records.append({
                'model': model_prefix,
                'unique_id': uid,
                'smape': round(s, 2),
            })

    report = pd.DataFrame(records)
    return report


model_prefixes = ['NHITS', 'MLP_baseline']
report = model_selection_report(cv_results, model_prefixes, cv_time)

# Pivot for easy comparison
pivot = report.pivot(index='unique_id', columns='model', values='smape')

print('Model Selection Report — sMAPE by Product:')
print(pivot.to_string())

# Overall recommendation
mean_smape = report.groupby('model')['smape'].mean()
print(f'\nMean sMAPE across all products:')
print(mean_smape.sort_values().to_string())

winner = mean_smape.idxmin()
print(f'\nRecommended model: {winner}')

In [ ]:
# Visual comparison: sMAPE per product, per model
if len(pivot.columns) >= 2 and len(pivot) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))

    x = np.arange(len(pivot))
    width = 0.35
    cols = pivot.columns.tolist()

    bars1 = ax.bar(x - width/2, pivot[cols[0]], width, label=cols[0], color='steelblue')
    if len(cols) > 1:
        bars2 = ax.bar(x + width/2, pivot[cols[1]], width, label=cols[1], color='orange')

    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=30, ha='right')
    ax.set_title('Model Comparison — sMAPE by Product (lower is better)')
    ax.set_ylabel('sMAPE (%)')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print('Model comparison: only one model available in CV results.')

## Modify This: Experiment with Scaling Parameters

Run the cells below to explore different configurations.

In [ ]:
# MODIFY: How does the forecast horizon affect accuracy?
# Try forecasting 3, 7, or 14 days ahead.

HORIZONS = [3, 7, 14]  # <-- change these

horizon_results = []
baguette_only = df_all[df_all['unique_id'] == 'BAGUETTE'].copy()

for h in HORIZONS:
    m = NHITS(
        h=h,
        input_size=3 * h,
        loss=MQLoss(quantiles=[0.1, 0.5, 0.8, 0.9]),
        max_steps=200,
        random_seed=42,
    )
    nf_h = NeuralForecast(models=[m], freq='D')
    cv_h = nf_h.cross_validation(df=baguette_only, n_windows=2, step_size=h)

    q50 = [c for c in cv_h.columns if 'q-0.5' in c][0]
    s = compute_smape(cv_h, q50)
    horizon_results.append({'horizon': h, 'smape': s})
    print(f'Horizon={h}: sMAPE={s:.2f}%')

print('\nObservation: longer horizons generally have higher sMAPE.')
print('This is expected — uncertainty compounds over longer time spans.')

In [ ]:
# MODIFY: How does input_size (lookback window) affect accuracy?
# The model looks back input_size days to forecast H days ahead.

INPUT_SIZES = [H, 2*H, 3*H, 5*H]  # 7, 14, 21, 35 days lookback

window_results = []

for isize in INPUT_SIZES:
    m = NHITS(
        h=H,
        input_size=isize,  # <-- this is what we are varying
        loss=MQLoss(quantiles=[0.1, 0.5, 0.8, 0.9]),
        max_steps=200,
        random_seed=42,
    )
    nf_i = NeuralForecast(models=[m], freq='D')
    cv_i = nf_i.cross_validation(df=baguette_only, n_windows=2, step_size=H)

    q50 = [c for c in cv_i.columns if 'q-0.5' in c][0]
    s = compute_smape(cv_i, q50)
    window_results.append({'input_size': isize, 'lookback_days': isize, 'smape': s})
    print(f'input_size={isize} ({isize}d lookback): sMAPE={s:.2f}%')

print('\nObservation: for bakery data with weekly seasonality,')
print('input_size = 3*H (21 days, 3 weeks) often works well —')
print('it captures the weekly pattern three times.')

## Summary

**What we demonstrated:**

1. **Global models** — one NHITS fit covers all products simultaneously, learning shared seasonality.
2. **Batch forecasting** — `forecast_in_batches()` handles large catalogs without memory issues.
3. **Cross-validation** — `nf.cross_validation()` gives honest out-of-sample accuracy estimates.
4. **Model selection** — Systematic sMAPE comparison justifies the model choice to stakeholders.
5. **Profiling** — Training time vs accuracy curves identify the minimum compute budget.

**Key production insight:** For the French Bakery, a single global NHITS model trained in under 60 seconds provides 7-day forecasts for all products, with cross-validated accuracy you can report to stakeholders.

**Next steps:**
- Exercise: `exercises/01_production_exercises.py` — build and verify a pipeline with shape checks
- Guide 2: `02_neuralforecast_patterns.md` — GPU training, logging, and fallback models